In [6]:
# libraries for the project

#import numpy as np
import gurobipy as gp
from gurobipy import GRB

import matplotlib.pyplot as plt
import json


In [58]:
# reading the json file from the test set

# we'll start easy by reading the first one

#  reading

filename = 'test01.json'

with open('test_data/'+filename, 'r') as file:
    data = json.load(file)

# creating the data structures

total_days= data['days']
total_shifts = data['shift_types']
skill_levels = data['skill_levels']
age_groups = data['age_groups']

age_to_number = {age: i + 1 for i, age in enumerate(age_groups)}   # dic
list_age = [age_to_number[age] for age in age_groups]    # list

# create a dic for occupants

occupants = data['occupants']

# create a dic for patients

patients = data['patients']

# create a dic for surgeons

surgeons = data['surgeons']

# create a dic for operating theaters

operating_theaters = data['operating_theaters']

# create a dic for the nurses

nurses = data['nurses']

# create a dic for rooms

rooms = data['rooms']

# saving the weights

weights = data['weights']
print(weights)

{'room_mixed_age': 5, 'room_nurse_skill': 1, 'continuity_of_care': 5, 'nurse_eccessive_workload': 1, 'open_operating_theater': 30, 'surgeon_transfer': 1, 'patient_delay': 5, 'unscheduled_optional': 150}


In [59]:
# modifying the gender using 0 and 1

print(occupants)


for occupant in occupants:
    if occupant['gender'] == 'A':
        occupant['gender'] = 0
    else:
        occupant['gender'] = 1

    occupant['age_group'] = age_to_number.get(occupant['age_group'])
   

print(occupants)

print(patients)

for patient in patients:
    if patient['gender'] == 'A':
        patient['gender'] = 0
    else:
        patient['gender'] = 1

    if patient['mandatory'] == False:
        patient['mandatory'] = 0
    else:
        patient['mandatory'] = 1

    patient['age_group'] = age_to_number.get(patient['age_group'])


for patient in patients:
    #(patient['incompatible_room_ids'])
    patient['compatible_rooms_ids'] = [room_id for room_id in rooms_id if room_id not in patient['incompatible_room_ids']]


print(patients)

[{'id': 'a0', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 4, 'workload_produced': [3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1], 'skill_level_required': [1, 0, 0, 0, 2, 0, 1, 1, 1, 1, 0, 0], 'room_id': 'r4'}, {'id': 'a1', 'gender': 'B', 'age_group': 'elderly', 'length_of_stay': 1, 'workload_produced': [1, 3, 1], 'skill_level_required': [1, 1, 0], 'room_id': 'r1'}, {'id': 'a2', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 5, 'workload_produced': [2, 2, 1, 2, 2, 1, 2, 1, 1, 3, 3, 1, 1, 2, 1], 'skill_level_required': [2, 2, 0, 1, 1, 0, 0, 2, 0, 1, 0, 0, 0, 0, 0], 'room_id': 'r0'}, {'id': 'a3', 'gender': 'B', 'age_group': 'adult', 'length_of_stay': 2, 'workload_produced': [3, 2, 1, 2, 3, 1], 'skill_level_required': [2, 1, 1, 1, 1, 1], 'room_id': 'r2'}, {'id': 'a4', 'gender': 'A', 'age_group': 'elderly', 'length_of_stay': 2, 'workload_produced': [1, 1, 1, 2, 2, 1], 'skill_level_required': [1, 0, 1, 2, 0, 0], 'room_id': 'r3'}, {'id': 'a5', 'gender': 'B', 'age_group': 'elderly

In [61]:
# adding a new feature to the patients, that tells us the id of the rooms where he can stay

# selecting all the rooms id

rooms_id = [room['id'] for room in rooms]
patients_id = [patient['id'] for patient in patients]
surgeons_id = [surgeon['id'] for surgeon in surgeons]
nurses_id = [nurse['id'] for nurse in nurses]
theaters_id = [theater['id'] for theater in operating_theaters]

print(rooms_id)
print(patients_id)
print(surgeons_id)
print(nurses_id)
print(theaters_id)



['r0', 'r1', 'r2', 'r3', 'r4']
['p00', 'p01', 'p02', 'p03', 'p04', 'p05', 'p06', 'p07', 'p08', 'p09', 'p10', 'p11', 'p12', 'p13', 'p14', 'p15', 'p16', 'p17', 'p18', 'p19', 'p20', 'p21', 'p22', 'p23', 'p24', 'p25', 'p26', 'p27', 'p28', 'p29', 'p30', 'p31', 'p32', 'p33', 'p34', 'p35', 'p36', 'p37', 'p38', 'p39', 'p40', 'p41']
['s0']
['n00', 'n01', 'n02', 'n03', 'n04', 'n05', 'n06', 'n07', 'n08', 'n09', 'n10', 'n11', 'n12']
['t0', 't1']


In [62]:
# construction of the model

Hosp = gp.Model("Hospital")

# ADDING VARIABLES

# dimensions of the variables:

R = len(rooms)
P = len(patients)
N = len(nurses)
S = len(surgeons)
O = len(operating_theaters)
T = total_days
D = len(total_shifts)

print("rooms:", R)
print("patients:", P)
print("nurses:", N)
print("surgeons:", S)
print("operating_theaters:", O)
print("total days:", T)
print("total_shifts:", D)

# variable x: if the patient i is in room r in time t

x = Hosp.addVars(rooms_id, patients_id, range(T), vtype=GRB.BINARY, name="x")  

# variable g: it represents the gender of the room r in the day t

g = Hosp.addVars(rooms_id, range(T), vtype=GRB.BINARY, name="g")

# variable Delta: it represents the difference between the maximum age in a room r and the minimum age in the same room in time t

Delta = Hosp.addVars(rooms_id, range(T), vtype=GRB.INTEGER, name="Delta")

# variable k: it represents that the nurse n is at the service of the patient i in time t at shift d and room r

k = Hosp.addVars(nurses_id, patients_id, range(T), total_shifts, rooms_id, vtype=GRB.BINARY, name="k")

# variable delta: it represents the difference between the maximum skill level of a nurse n and the level of the patient i in time t at shift d in room r

delta = Hosp.addVars(nurses_id, patients_id, range(T), total_shifts, rooms_id, vtype=GRB.INTEGER, name="delta")

# variable z: it represents that the surgeon s is operating the patient i in time t in the theater o

z = Hosp.addVars(surgeons_id, patients_id, theaters_id, range(T), vtype=GRB.BINARY, name="z")

# variable eps: it represents a bound over the number of theaters o tha can be used in a day t

eps = Hosp.addVars(theaters_id, range(T), vtype=GRB.INTEGER, name="eps")

# variable gamma: it represents the a bound over the number of theaters that a surgeon s can have in a day t

gamma = Hosp.addVars(surgeons_id, range(T), vtype=GRB.INTEGER, name="gamma")

# variable sigma: it represents the day in which the patient i enters the hospital (>= R_i)

sigma = Hosp.addVars(patients_id, vtype=GRB.INTEGER, name="sigma")



rooms: 5
patients: 42
nurses: 13
surgeons: 1
operating_theaters: 2
total days: 21
total_shifts: 3


In [ ]:
# adding the constraints to the model


# H5:

# definding a variable that maps the instant in which the patient i is in the hospital

instant_patient = {}  # onedimensional dictionary
for patient in patients:
    instant_patient[patient['id']] = (1-patient['mandatory'])*patient['surgery_release_day'] + patient['mandatory']*sigma[patient['id']]

# H1

for room in rooms:
    for t in range(T):
        Hosp.addConstr(gp.quicksum(x[room['id'], patient['id'], t] * patient['gender'] for patient in patients) <= g[room['id'], t]*gp.quicksum(x[room['id'], patient['id'], t] for patient in patients), name="H1_1")
        Hosp.addConstr(gp.quicksum(x[room['id'], patient['id'], t] *(1 - patient['gender']) for patient in patients) <= (1-g[room['id'], t])*gp.quicksum(x[room['id'], patient['id'], t] for patient in patients), name="H1_1")




# H2

# creating a variable that is a matrix patients x rooms, there's a 1 if the patient i can stay in the room r

y = {}  # bidimensional dictionary

for patient in patients:
    for room_id in rooms_id:
        y[(patient['id'], room_id)] = 1 if room_id in patient['compatible_rooms_ids'] else 0

#print(y)

for patient in patients:                     # da sistemare 
    for t in range(T):
        room_assignments = gp.quicksum(x[room['id'], patient['id'], t] * y[(patient['id'], room['id'])] 
                                     for room in rooms)
        
        # Aggiungiamo la presenza del paziente come vincolo
        Hosp.addConstr(
            room_assignments <= 1,
            name=f"H2_upper_{patient['id']}_{t}"
        )
        
        # Se il paziente dovrebbe essere presente (periodo di degenza), 
        # room_assignments deve essere esattamente 1
        mandatory_stay = True  # Assumo che sia obbligatorio, modificalo in base alle tue esigenze
        if mandatory_stay:
            Hosp.addConstr(
                room_assignments >= 1,
                name=f"H2_lower_{patient['id']}_{t}"
            )



# H7

# creating a variable that maps the stay of an occupant in the hospital, it is 1 until he stays there 

stay_occupant = {}  # tridimensional dictionary

for occupant in occupants:
    for t in range(T):
        for room_id in rooms_id:
            stay_occupant[(occupant['id'],room_id, t)] = 1 if occupant['length_of_stay'] >= t and occupant['room_id'] == room_id else 0

#print(stay_occupant)

for room in rooms:
    for t in range(T):
        Hosp.addConstr(gp.quicksum(x[room['id'], patient['id'], t] for patient in patients) + 
                       gp.quicksum(stay_occupant[(occupant['id'], room['id'], t)] for occupant in occupants) <= room['capacity'], name="H7")
        





# S1:

# da scrivere




# H6:

for patient in patients:
    if not patient['mandatory']:
        Hosp.addConstr(sigma[patient['id']] == patient['surgery_release_day'], name="H6_1")
    else:
        Hosp.addConstr(sigma[patient['id']] >= patient['surgery_release_day'], name="H6_1")
        Hosp.addConstr(sigma[patient['id']] <= patient['surgery_due_day'], name="H6_2")





# S2:

# creating a variable that maps the skill level of patient i in time t shift d

skill_level_patient = {}  # tridimensional dictionary

for patient in patients:
    for i,shift_day in enumerate(patient['skill_level_required']):
        if i%3==0:
            shift= 'early'
        elif i%3==1:
            shift= 'late'
        else:
            shift= 'night'
        t = i//3
        skill_level_patient[(patient['id'], t, shift)] = patient['skill_level_required'][i]

#print(skill_level_patient)
for t in range(T):
    for room in rooms:
        for shift in total_shifts:
            for nurse in nurses:
                for patient in patients:
                    # Use separate variable for readability
                    start_time = instant_patient[patient['id']]
                    stay_length = patient['length_of_stay']
                    
                    # Use multiplication instead of boolean logic
                    # This will be 1 when the patient is present, 0 otherwise
                    presence_indicator = (t >= start_time) * (t <= start_time + stay_length)
                    
                    Hosp.addConstr(
                        nurse['skill_level'] * k[nurse['id'], patient['id'], t, shift, room['id']] +
                        delta[nurse['id'], patient['id'], t, shift, room['id']] >=
                        skill_level_patient.get((patient['id'], t - start_time, shift), 0) * 
                        x[room['id'], patient['id'], t] * presence_indicator,
                        name=f"S2_1_{nurse['id']}_{patient['id']}_{t}_{shift}_{room['id']}"
                    )

TypeError: unsupported operand type(s) for *: 'TempConstr' and 'TempConstr'